# **Transaction NER**

You are given `train.jsonl` and `val.jsonl` (tokens + `ner_tags`) and `test.jsonl` (tokens only).

For each sample in `train.jsonl` and `val.jsonl` the ID of the annotator has also been attached. A total of 5 annotators have worked on this task, and each and every sample has been annotated by at least two annotators.   


**Keep in mind that annotators are also humans and can make mistakes.**


Train a token classifier and, for every test transaction, output the extracted
**counterparty, processor, transaction_method, bank_service_event, recurring_flag** as JSON.

This notebook is a **plain baseline**: it trains on the data exactly as given and writes
`predictions.json`. It is your starting point, not the answer — improving the score is up to you.
Runtime: a GPU is recommended.

## **INSTRUCTIONS GIVEN TO ANNOTATORS** [IMPORTANT]

For each transaction you have to annotate the following:
1. The "Counter Party". This is the party with which the transaction has taken place. It can be the party to which the money was paid to, or it can be the party which gave the money to you. Depends on the direction of the transactions
2. "Transaction Method" and "Processor" are to be marked based on your understanding.
3. For standalone bank events like "withdrawal", "overdraft", "fee" etc, it should be marked as "Bank Service Events".
4. Any indicator that the transaction is a `recurring`/`pre authorized`/`subscription`/`auto debit`/`auto pay`/`auto approved` etc should be marked as "Recurring Flag" as it indicates that this transaction can happen without explicit approval at regular intervals.
5. In case you are unsure of any token, just mark it as "O"

In [ ]:
# ================================================================
# IMPORTANT!!! Fill in your details first
NAME = 'Disha'
EMAIL = 'dishagomes2005@gmail.com'
ROLL_NUMBER = '230953010'
COLLEGE_NAME = 'Manipal Institute of Technology'
# ================================================================


In [ ]:
!pip install -q transformers datasets seqeval torch evaluate tqdm

In [ ]:
import os
import time
import json
import torch
import evaluate
import requests
import numpy as np
from tqdm import tqdm
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    DataCollatorForTokenClassification, TrainingArguments, Trainer
)

MODEL            = "jhu-clsp/ettin-encoder-32m" # DO NOT CHANGE THE MODEL!
MAX_LEN          = 64
BATCH_SIZE       = 128
TRAIN, VAL, TEST = "datasets/train.jsonl", "datasets/val.jsonl", "datasets/test.jsonl"
BASE_URL         = "http://3.6.116.106:8990"

# Fixed label set for this task
LABELS = [
    "O",
    "I-COUNTERPARTY_NAME",
    "I-PROCESSOR",
    "I-TRANSACTION_METHOD",
    "I-BANK_SERVICE_EVENT",
    "I-RECURRING_FLAG",
    "I-FILLER_WORD",
    "I-SEPARATOR_PUNCTUATION"
]

label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

device   = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu": print("Google Colab offers GPU environment. Make sure to use it for faster training")

In [ ]:
def download_dataset(filename: str, out_path: str | None = None) -> None:
    url = f"{BASE_URL}/datasets/{filename}"
    out_path = out_path or filename

    if os.path.exists(out_path):
        print('File already exists at the desired location. Skipping...')
        return

    out_path_obj = Path(out_path)
    if out_path_obj.parent:
        out_path_obj.parent.mkdir(parents=True, exist_ok=True)

    backoff_delays = [3, 5, 10]
    total_attempts = len(backoff_delays) + 1

    for attempt in range(total_attempts):
        try:
            resp = requests.get(url, timeout=60)
            resp.raise_for_status()
            break
        except requests.RequestException as e:
            if attempt < len(backoff_delays):
                delay = backoff_delays[attempt]
                print(f"Attempt {attempt + 1} failed ({e}). Retrying in {delay} seconds...")
                time.sleep(delay)
            else:
                print(f"All {total_attempts} attempts failed.")
                raise e

    with open(out_path, "wb") as f:
        f.write(resp.content)

    n_lines = resp.text.count("\n")
    print(f"Saved {out_path}  ({len(resp.content):,} bytes, {n_lines:,} lines)")


download_dataset('train.jsonl', TRAIN)
download_dataset('val.jsonl', VAL)
download_dataset('test.jsonl', TEST)

In [ ]:
def read(path):
    rows = [json.loads(l) for l in open(path)]
    return Dataset.from_list(rows)

train_ds, val_ds = read(TRAIN), read(VAL)
print(train_ds, "\nexample:", json.dumps(train_ds[0], indent=4))

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL)

def encode(batch):
    enc = tok(
        batch["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LEN
    )
    all_labels = []

    for i, tags in enumerate(batch["ner_tags"]):
        word_ids, prev, labels = enc.word_ids(i), None, []
        for w in word_ids:
            if w is None:   labels.append(-100)                      # special tokens
            elif w != prev: labels.append(label2id.get(tags[w], 0))  # first subword
            else:           labels.append(-100)                      # later subwords
            prev = w
        all_labels.append(labels)
    enc["labels"] = all_labels
    return enc

keep = ["input_ids", "attention_mask", "labels"]
train_enc = train_ds.map(encode, batched=True, remove_columns=train_ds.column_names)
val_enc   = val_ds.map(encode,   batched=True, remove_columns=val_ds.column_names)

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL, num_labels=len(LABELS), id2label=id2label, label2id=label2id
)

metric = evaluate.load("seqeval")

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    true_lab, true_pred = [], []
    for pred, lab in zip(preds, p.label_ids):
        true_lab.append([id2label[l] for l in lab if l != -100])
        true_pred.append([id2label[pr] for pr, l in zip(pred, lab) if l != -100])
    r = metric.compute(predictions=true_pred, references=true_lab, zero_division=0)
    return {
        "f1": r["overall_f1"],
        "precision": r["overall_precision"],
        "recall": r["overall_recall"]
    }

args = TrainingArguments(
    output_dir="ckpt",
    learning_rate=1e-4,
    num_train_epochs=5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=100,
    report_to="none"
)

trainer = Trainer(model, args, train_dataset=train_enc, eval_dataset=val_enc,
                  data_collator=DataCollatorForTokenClassification(tok),
                  compute_metrics=compute_metrics)
trainer.train()

## Inference → per-transaction JSON

For each test transaction we predict a tag per word (first-subword rule) and join the
tokens of each target class into a field. This is what gets scored on the server.

In [ ]:
model.eval()
FIELD = {"COUNTERPARTY_NAME": "counterparty", "PROCESSOR": "processor",
         "TRANSACTION_METHOD": "transaction_method", "BANK_SERVICE_EVENT": "bank_service_event",
         "RECURRING_FLAG": "recurring_flag"}

@torch.no_grad()
def tag_words_batch(batch_tokens):
    enc = tok(
        batch_tokens,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LEN
    ).to(device)
    preds = model(**enc).logits.argmax(-1).tolist()

    batch_out = []
    for i, tokens in enumerate(batch_tokens):
        out, seen = ["O"] * len(tokens), set()
        for idx, w in enumerate(enc.word_ids(i)):
            if w is not None and w not in seen:
                out[w] = id2label[preds[i][idx]]; seen.add(w)
        batch_out.append(out)
    return batch_out

def extract(tokens, tags):
    rec = {v: [] for v in FIELD.values()}
    for tok, tag in zip(tokens, tags):
        b = tag[2 : ] if tag.startswith("I-") else "O"
        if b in FIELD: rec[FIELD[b]].append(tok)

    return {k: " ".join(v) for k, v in rec.items()}

records = [json.loads(line) for line in open(TEST)]
results = []

for i in tqdm(range(0, len(records), BATCH_SIZE), desc='Test Inference'):
    chunk = records[i : i + BATCH_SIZE]
    tags_batch = tag_words_batch([r["tokens"] for r in chunk])
    for r, tags in zip(chunk, tags_batch):
        results.append({"id": r["id"], **extract(r["tokens"], tags)})

json.dump(results, open("predictions.json", "w"), indent=2)
print("wrote", len(results), "rows ->", "predictions.json")
print(results[:3])

In [ ]:
def _print_error(resp) -> None:
    print(f"Request failed: {resp.status_code} {resp.reason}")

    try:
        detail = resp.json().get("detail", resp.text)
    except ValueError:
        print(resp.text)
        return

    if isinstance(detail, dict):
        header = detail.get("error", "Validation failed")
        print(header)
        problems = detail.get("problems", [])
        if isinstance(problems, list):
            for p in problems:
                print(f"  - {p}")
        else:
            print(f"  {problems}")
        for k, v in detail.items():
            if k not in ("error", "problems"):
                print(f"  {k}: {v}")
    else:
        print(detail)


def submit_predictions(pred_path: str, name: str, email: str, roll_number: str, college: str) -> dict:
    url = f"{BASE_URL}/submit"
    with open(pred_path, "rb") as f:
        files = {"predictions": (pred_path, f, "application/json")}
        data = {
            "name": name,
            "email": email,
            "roll_number": roll_number,
            "college": college
        }
        resp = requests.post(url, data=data, files=files, timeout=120)

    if not resp.ok:
        _print_error(resp)
        return {}

    return resp.json()


def print_results(result: dict) -> None:
    if not result:
        return

    per_field = result["per_field"]

    headers = ["field", "metric", "f1", "precision", "recall", "exact", "support"]
    rows = []
    for field, m in per_field.items():
        rows.append([
            field,
            m["metric"],
            f'{m["f1"]:.4f}',
            f'{m["precision"]:.4f}',
            f'{m["recall"]:.4f}',
            f'{m["exact"]:.4f}',
            str(m["support"]),
        ])

    widths = [max(len(h), *(len(r[i]) for r in rows)) for i, h in enumerate(headers)]

    def fmt_row(cells):
        return "  ".join(c.ljust(w) for c, w in zip(cells, widths))

    print()
    print(fmt_row(headers))
    print("  ".join("-" * w for w in widths))
    for r in rows:
        print(fmt_row(r))
    print()

    print(f'This submission macro-F1 : {result["this_submission_macro_f1"]:.4f}')
    print(f'Best macro-F1 so far      : {result["best_macro_f1"]:.4f}  (at {result["best_at"]})')
    print(f'Attempts so far           : {result["attempts"]}')
    print(f'Attempts remaining        : {result["attempts_remaining"]}')

In [ ]:
print_results(submit_predictions(
    'predictions.json',
    NAME,
    EMAIL,
    ROLL_NUMBER,
    COLLEGE_NAME
))